In [1]:
import sys

!{sys.executable} -m pip install -q ccxt pyarrow pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Users/nguyencaogiakhanh/miniconda3/bin/python3 -m pip install --upgrade pip


In [2]:
import ccxt
import pandas as pd
import time
from datetime import datetime, timezone
from pathlib import Path

# ── CONFIGURATION ────────────────────────────────────────────────────────────
ASSETS     = ["BTC", "ETH", "SOL"]
EXCHANGES  = ["binance", "bybit", "okx"]

START_DATE = "2020-01-01T00:00:00Z"
END_DATE   = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
TIMEFRAME  = "1h"

SPREAD_BPS  = 2.0
HALF_SPREAD = SPREAD_BPS / 10_000

EXCHANGE_CONSTANTS = {
    "binance": {"funding_interval_hours": 8,  "perp_taker_fee_bps": 5.0,  "spot_taker_fee_bps": 10.0},
    "bybit":   {"funding_interval_hours": 8,  "perp_taker_fee_bps": 5.5,  "spot_taker_fee_bps": 10.0},
    "okx":     {"funding_interval_hours": 8,  "perp_taker_fee_bps": 5.0,  "spot_taker_fee_bps": 8.0},
}

NO_SPOT_EXCHANGES = set()

OUT_DIR = Path("./crypto_hourly")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Saving to:", OUT_DIR.resolve())
print("End date :", END_DATE)
print("Exchanges:", EXCHANGES)
print("Assets   :", ASSETS)

Saving to: /Users/nguyencaogiakhanh/Desktop/quanttradingstrats/crypto_hourly
End date : 2026-03-03T20:11:12Z
Exchanges: ['binance', 'bybit', 'okx']
Assets   : ['BTC', 'ETH', 'SOL']


In [3]:
def make_exchange(exchange_id: str, market_type: str):
    """
    Create a ccxt exchange instance.
    market_type: 'swap' for perps, 'spot' for spot.
    """
    opts = {"enableRateLimit": True, "options": {"defaultType": market_type}}

    if exchange_id == "binance":
        return ccxt.binance(opts)

    if exchange_id == "bybit":
        if market_type == "swap":
            opts["options"]["fetchMarkets"] = ["linear"]
        else:
            opts["options"]["fetchMarkets"] = ["spot"]
        return ccxt.bybit(opts)

    if exchange_id == "okx":
        return ccxt.okx(opts)

    raise ValueError(f"Unsupported exchange: {exchange_id}")


def symbols(exchange_id: str, asset: str):
    """
    Return (perp_symbol, spot_symbol) in ccxt unified format.
    All three exchanges (binance, bybit, okx) use USDT pairs.
    """
    asset = asset.upper()
    perp = f"{asset}/USDT:USDT"
    spot = f"{asset}/USDT"
    return perp, spot

In [4]:
def fetch_ohlcv(exchange, symbol, timeframe, start_date, end_date, limit=1000, params=None, verbose=True):
    params = params or {}
    since = exchange.parse8601(start_date)
    end_ts = exchange.parse8601(end_date)
    rows = []

    while True:
        batch = exchange.fetch_ohlcv(symbol, timeframe, since=since, limit=limit, params=params)
        if not batch:
            break

        batch = [r for r in batch if r[0] <= end_ts]
        if not batch:
            break

        rows.extend(batch)
        last_ts = rows[-1][0]

        if verbose:
            print(f"  [{exchange.id}:{symbol}] up to {pd.to_datetime(last_ts, unit='ms', utc=True)} | rows={len(rows)}")

        if last_ts >= end_ts or len(batch) < limit:
            break

        since = last_ts + 1
        time.sleep(exchange.rateLimit / 1000)

    df = pd.DataFrame(rows, columns=["timestamp", "open", "high", "low", "close", "volume"])
    if df.empty:
        return df
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    return df.set_index("timestamp")


def fetch_funding(exchange, symbol, start_date, end_date, limit=1000, verbose=True):
    if not exchange.has.get("fetchFundingRateHistory", False):
        print(f"  [funding] {exchange.id} does not support funding history via ccxt.")
        return pd.DataFrame(columns=["funding_rate_raw"])

    since = exchange.parse8601(start_date)
    end_ts = exchange.parse8601(end_date)
    rows = []

    while True:
        try:
            batch = exchange.fetch_funding_rate_history(symbol, since=since, limit=limit)
        except Exception as e:
            print(f"  [funding] {exchange.id}:{symbol} failed: {e}")
            break

        if not batch:
            break

        batch = [r for r in batch if r.get("timestamp") is not None and r["timestamp"] <= end_ts]
        if not batch:
            break

        rows.extend(batch)
        last_ts = rows[-1]["timestamp"]

        if verbose:
            print(f"  [funding {exchange.id}] up to {pd.to_datetime(last_ts, unit='ms', utc=True)} | rows={len(rows)}")

        if last_ts >= end_ts or len(batch) < limit:
            break

        since = last_ts + 1
        time.sleep(exchange.rateLimit / 1000)

    if not rows:
        return pd.DataFrame(columns=["funding_rate_raw"])

    df = pd.DataFrame(
        {
            "timestamp": [pd.to_datetime(r["timestamp"], unit="ms", utc=True) for r in rows],
            "funding_rate_raw": [float(r["fundingRate"]) for r in rows],
        }
    ).set_index("timestamp")
    return df


def fetch_open_interest(exchange, symbol, start_date, end_date, timeframe="1h", limit=1000, verbose=True):
    if not exchange.has.get("fetchOpenInterestHistory", False):
        print(f"  [OI] {exchange.id} does not support OI history via ccxt.")
        return pd.DataFrame(columns=["open_interest_usd"])

    since = exchange.parse8601(start_date)
    end_ts = exchange.parse8601(end_date)
    rows = []

    while True:
        try:
            batch = exchange.fetch_open_interest_history(symbol, timeframe, since=since, limit=limit)
        except Exception as e:
            print(f"  [OI] {exchange.id}:{symbol} failed: {e}")
            break

        if not batch:
            break

        batch = [r for r in batch if r.get("timestamp") is not None and r["timestamp"] <= end_ts]
        if not batch:
            break

        rows.extend(batch)
        last_ts = rows[-1]["timestamp"]

        if verbose:
            print(f"  [OI {exchange.id}] up to {pd.to_datetime(last_ts, unit='ms', utc=True)} | rows={len(rows)}")

        if last_ts >= end_ts or len(batch) < limit:
            break

        since = last_ts + 1
        time.sleep(exchange.rateLimit / 1000)

    if not rows:
        return pd.DataFrame(columns=["open_interest_usd"])

    def oi_value(r):
        for k in ("openInterestValue", "openInterest", "value"):
            if k in r and r[k] is not None:
                return float(r[k])
        return float("nan")

    df = pd.DataFrame(
        {
            "timestamp": [pd.to_datetime(r["timestamp"], unit="ms", utc=True) for r in rows],
            "open_interest_usd": [oi_value(r) for r in rows],
        }
    ).set_index("timestamp")
    return df

In [5]:
def fetch_exchange_asset(
    exchange_id: str,
    asset: str,
    start_date: str = START_DATE,
    end_date: str   = END_DATE,
    timeframe: str  = TIMEFRAME,
) -> pd.DataFrame:
    """
    Fetch hourly data for one (exchange, asset) pair.
    Returns a DataFrame with columns suffixed by exchange name,
    aligned to a full hourly UTC index from start_date to end_date.
    """
    ex_suffix = exchange_id.lower()
    perp_symbol, spot_symbol = symbols(exchange_id, asset)
    has_spot = exchange_id not in NO_SPOT_EXCHANGES

    print(f"\n{'='*55}")
    print(f"  {exchange_id.upper()} — {asset}  |  {start_date} → {end_date}")
    print(f"{'='*55}")

    # ── 1. Perp OHLCV ────────────────────────────────────────────────────────
    print(f"  [1/4] Perp OHLCV  ({perp_symbol}) ...")
    perp_ex = make_exchange(exchange_id, "swap")
    perp_ohlcv = fetch_ohlcv(perp_ex, perp_symbol, timeframe, start_date, end_date, verbose=False)
    if perp_ohlcv.empty:
        print("  ⚠ No perp OHLCV returned — skipping this exchange/asset.")
        return pd.DataFrame()

    # ── 2. Spot OHLCV (skip if exchange has no spot, or if geo-blocked) ───────
    spot_ohlcv = pd.DataFrame()
    if has_spot:
        print(f"  [2/4] Spot OHLCV  ({spot_symbol}) ...")
        try:
            spot_ex = make_exchange(exchange_id, "spot")
            spot_ohlcv = fetch_ohlcv(spot_ex, spot_symbol, timeframe, start_date, end_date, verbose=False)
        except Exception as e:
            print(f"  ⚠ Spot fetch failed (geo-block or unsupported): {e}")
            print(f"    → spot columns for {ex_suffix} will be NaN")
    else:
        print(f"  [2/4] Spot OHLCV  — skipped ({exchange_id} has no spot market)")

    # ── 3. Funding rates ──────────────────────────────────────────────────────
    print(f"  [3/4] Funding rates ...")
    funding = fetch_funding(perp_ex, perp_symbol, start_date, end_date, verbose=False)

    # ── 4. Open interest ──────────────────────────────────────────────────────
    print(f"  [4/4] Open interest ...")
    oi = fetch_open_interest(perp_ex, perp_symbol, start_date, end_date,
                             timeframe=timeframe, verbose=False)

    # ── ASSEMBLE into a single hourly DataFrame ──────────────────────────────
    full_idx = pd.date_range(
        start=pd.to_datetime(start_date),
        end=pd.to_datetime(end_date),
        freq="1h",
        tz="UTC",
    )
    df = pd.DataFrame(index=full_idx)
    df.index.name = "timestamp"

    # Mark price = perp candle close, reindexed to full hourly grid
    df[f"mark_price_{ex_suffix}"] = perp_ohlcv["close"].reindex(full_idx)
    df[f"best_bid_{ex_suffix}"]   = df[f"mark_price_{ex_suffix}"] * (1 - HALF_SPREAD)
    df[f"best_ask_{ex_suffix}"]   = df[f"mark_price_{ex_suffix}"] * (1 + HALF_SPREAD)

    # Spot columns — NaN if no spot data available
    if not spot_ohlcv.empty:
        spot_close = spot_ohlcv["close"].reindex(full_idx)
        df[f"spot_best_bid_{ex_suffix}"] = spot_close * (1 - HALF_SPREAD)
        df[f"spot_best_ask_{ex_suffix}"] = spot_close * (1 + HALF_SPREAD)
    else:
        df[f"spot_best_bid_{ex_suffix}"] = float("nan")
        df[f"spot_best_ask_{ex_suffix}"] = float("nan")

    # Funding rate — forward-fill across the funding interval
    if not funding.empty:
        fr = funding["funding_rate_raw"].reindex(full_idx)
        df[f"funding_rate_raw_{ex_suffix}"] = fr.ffill()
    else:
        df[f"funding_rate_raw_{ex_suffix}"] = float("nan")

    # Open interest — forward-fill gaps
    if not oi.empty:
        oiv = oi["open_interest_usd"].reindex(full_idx)
        df[f"open_interest_usd_{ex_suffix}"] = oiv.ffill()
    else:
        df[f"open_interest_usd_{ex_suffix}"] = float("nan")

    print(f"  ✅ {len(df)} rows assembled  |  non-null mark_price: {df[f'mark_price_{ex_suffix}'].notna().sum()}")
    return df

In [6]:
def save_parquet(df: pd.DataFrame, exchange_id: str, asset: str):
    """Save one exchange-asset DataFrame as a parquet file."""
    path = OUT_DIR / f"{exchange_id.lower()}_{asset.upper()}USDT_1h.parquet"
    df.to_parquet(path)
    print(f"  Saved: {path}")

# ── MAIN FETCH LOOP ──────────────────────────────────────────────────────────
for ex_id in EXCHANGES:
    for asset in ASSETS:
        try:
            df = fetch_exchange_asset(ex_id, asset)
            if df.empty:
                print(f"  ⚠ No data for {ex_id}/{asset} — skipping save.\n")
                continue
            save_parquet(df, ex_id, asset)
        except Exception as e:
            print(f"  ERROR fetching {ex_id}/{asset}: {e}\n")
            continue

print("\n" + "="*55)
print("All done. Files in:", OUT_DIR.resolve())
print("="*55)


  BINANCE — BTC  |  2020-01-01T00:00:00Z → 2026-03-03T20:11:12Z
  [1/4] Perp OHLCV  (BTC/USDT:USDT) ...
  [2/4] Spot OHLCV  (BTC/USDT) ...
  [3/4] Funding rates ...
  [4/4] Open interest ...
  [OI] binance:BTC/USDT:USDT failed: binance {"msg":"parameter 'startTime' is invalid.","code":-1130}
  ✅ 54093 rows assembled  |  non-null mark_price: 54093
  💾 Saved: crypto_hourly/binance_BTCUSDT_1h.parquet

  BINANCE — ETH  |  2020-01-01T00:00:00Z → 2026-03-03T20:11:12Z
  [1/4] Perp OHLCV  (ETH/USDT:USDT) ...
  [2/4] Spot OHLCV  (ETH/USDT) ...
  [3/4] Funding rates ...
  [4/4] Open interest ...
  [OI] binance:ETH/USDT:USDT failed: binance {"msg":"parameter 'startTime' is invalid.","code":-1130}
  ✅ 54093 rows assembled  |  non-null mark_price: 54093
  💾 Saved: crypto_hourly/binance_ETHUSDT_1h.parquet

  BINANCE — SOL  |  2020-01-01T00:00:00Z → 2026-03-03T20:11:12Z
  [1/4] Perp OHLCV  (SOL/USDT:USDT) ...
  [2/4] Spot OHLCV  (SOL/USDT) ...
  [3/4] Funding rates ...
  [4/4] Open interest ...
  [O